In [ ]:
# # ATAS v1.1 – Long and Short Pause Analysis Functions

# This notebook contains helper functions for post-processing ATAS outputs.

# It includes functions for:
# - extracting long and short pause durations from event-level CSV files
# - computing pause counts, mean durations, standard deviations, and coefficients of variation
# - merging participant-level metadata (CWS/CWNS details)
# - calculating speech rate
# - merging optional SSI and %SS scores

# This notebook is imported into:
# `Compute_long_and_short_pause_stats.ipynb`

# Ensure this file is located in the same project directory when running the analysis.

---

Developed as part of ATAS v1.1 – CWS/CWNS extension.

In [ ]:
def merge_dataframes_on_filename(df_all, PWNS_par, PWS_par):
    """
    Merge df_all with either PWNS_par or PWS_par based on matching file names,
    using extracted ID. Keep only matched rows, and structure the final DataFrame
    with all columns from PWNS/PWS appearing first (in original order).
    'File_Name' is dropped, 'ID' is kept.
    """

    df_all_copy = df_all.copy()

    # Extract ID from File_Name and add it as a new column
    #df_all_copy['ID'] = df_all_copy['File_Name'].apply(lambda x: x.rsplit('.', 1)[0].split('_')[0])
    df_all_copy['ID'] = df_all_copy['File_Name'].apply(lambda x: x.rsplit('.', 1)[0])

    # Store matched data rows
    merged_rows = []

    for i, row in df_all_copy.iterrows():
        participant_id = row['ID']

        match_pwns = PWNS_par[PWNS_par['ID'] == participant_id]
        match_pws = PWS_par[PWS_par['ID'] == participant_id]

        if not match_pwns.empty:
            merged_data = match_pwns.iloc[0].to_dict()  # PWNS match
            merged_rows.append({**merged_data, **row.drop('File_Name')})

        elif not match_pws.empty:
            merged_data = match_pws.iloc[0].to_dict()  # PWS match
            merged_rows.append({**merged_data, **row.drop('File_Name')})

        # Else: skip (unmatched row)

    # Create DataFrame from merged rows
    df_merged = pd.DataFrame(merged_rows)

    # Reorder columns: all columns from PWNS_par/PWS_par first, then the rest
    reference_columns = [col for col in PWNS_par.columns if col in df_merged.columns]  # assumes both have same structure
    remaining_columns = [col for col in df_merged.columns if col not in reference_columns]
    final_column_order = reference_columns + remaining_columns

    return df_merged[final_column_order]


In [ ]:
def process_pause_durations(file_need, df, i, pause_threshold):
    # Check if columns exist, if not, create them

    # Add all event durations sequence-wise
    if 'All_events_durations' not in df.columns:
        df['All_events_durations'] = None
    
    if 'long_p_durations' not in df.columns:
        df['long_p_durations'] = None
    if 'short_p_durations' not in df.columns:
        df['short_p_durations'] = None
    if 'long_p_count' not in df.columns:
        df['long_p_count'] = None
    if 'short_p_count' not in df.columns:
        df['short_p_count'] = None
        
    if 'long_p_durations_mean' not in df.columns:
        df['long_p_durations_mean'] = None
    if 'short_p_durations_mean' not in df.columns:
        df['short_p_durations_mean'] = None
    if 'long_p_durations_cv' not in df.columns:
        df['long_p_durations_cv'] = None
    if 'short_p_durations_cv' not in df.columns:
        df['short_p_durations_cv'] = None
    

    # Load the corresponding CSV file
    ddf = pd.read_csv(file_need)


    # Identify long and short pauses
    long_p = [idx for idx in ddf.index if (ddf["Labels"][idx] == 'p') & (ddf["Time_diff"][idx] > pause_threshold)]
    short_p = [idx for idx in ddf.index if (ddf["Labels"][idx] == 'p') & (ddf["Time_diff"][idx] <= pause_threshold)]
    

    # Calculate long pause durations, means, and CV
    long_p_dur = [ddf["Time_diff"][idx] for idx in long_p]
    if long_p_dur:  
        long_p_mean = statistics.mean(long_p_dur)
        long_p_cv = variation(long_p_dur)
    else:
        long_p_mean = 0
        long_p_cv = 0

    # Calculate short pause durations, means, and CV
    short_p_dur = [ddf["Time_diff"][idx] for idx in short_p]
    if short_p_dur: 
        short_p_mean = statistics.mean(short_p_dur)
        short_p_cv = variation(short_p_dur)
    else:
        short_p_mean = 0
        short_p_cv = 0
        
    all_events_dur = ddf["Time_diff"].values

    pause_type = np.where(
        (ddf['Labels'] == 'p') & (ddf['Time_diff'] >= pause_threshold), 2,   # First condition
        np.where(
            (ddf['Labels'] == 'p') & (ddf['Time_diff'] < pause_threshold), 1,  # Second condition
            0  
        )
    )

# Array corresponding to the conditions.


    # Update the corresponding row in df with the processed pause data
    df.at[i, 'All_events_durations'] = ','.join(map(str, all_events_dur))  # Convert to string
    df.at[i, 'long_p_durations'] = ','.join(map(str, long_p_dur))          # Convert to string
    df.at[i, 'short_p_durations'] = ','.join(map(str, short_p_dur))        # Convert to string
    df.at[i,'Event_type'] = ','.join(map(str, pause_type))        # Convert to string
    df.at[i, 'long_p_count'] = len(long_p_dur)
    df.at[i, 'short_p_count'] = len(short_p_dur)
    df.at[i, 'long_p_durations_mean'] = long_p_mean
    df.at[i, 'short_p_durations_mean'] = short_p_mean
    df.at[i, 'long_p_durations_cv'] = long_p_cv
    df.at[i, 'short_p_durations_cv'] = short_p_cv
    
def calculate_speech_rate(df_all_1):
    # Calculate speech rate for both AWS and AWNS participants
    speech_rate_1 = df_all_1["Final_word_count"] / df_all_1["Total_Duration_Clipped_s"] * 60
    df_all_1["Speech_Rate"] = speech_rate_1


In [ ]:
def merge_ssi_scores(df_all_1, ssi_scores_cws, ssi_scores_cwns):
    """
    Merge two separate SSI score files into df_all_1 using 'ID' as the key.
    Both SSI inputs contain complementary sets of columns.
    Assumes 'ID' is already present in df_all_1.
    """
    # Combine the two SSI score DataFrames on 'ID'
    ssi_combined = pd.concat([ssi_scores_cws, ssi_scores_cwns], ignore_index=True)

    # Merge the combined SSI scores with df_all_1 based on 'ID'
    df_all_merged = pd.merge(df_all_1, ssi_combined, on='ID', how='left')

    return df_all_merged
